# WAFER-GUARD · Fab-7 Excursion Detector

## 0 · Environment

In [ ]:
import os, random, copy, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, fbeta_score,
    average_precision_score, confusion_matrix, classification_report
)

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

Device: cpu
PyTorch: 2.11.0+cpu


## 1 · Data

In [ ]:
FILE_PATH = "/content/secom_telemetry_modified.csv"
if not os.path.exists(FILE_PATH):
    from google.colab import files
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if not csv_names:
        raise FileNotFoundError("Upload secom_telemetry_modified.csv")
    FILE_PATH = "/content/" + csv_names[0]

df = pd.read_csv(FILE_PATH)
required = {"label", "timestamp", "sensor_0", "sensor_1"}
missing_required = required - set(df.columns)
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
if df["timestamp"].isna().any():
    raise ValueError("Some timestamps could not be parsed; resolve them before chronological evaluation.")
if not set(df["label"].dropna().unique()).issubset({-1, 1}):
    raise ValueError("Expected labels -1 (pass) and +1 (defect).")
df = df.sort_values("timestamp", kind="mergesort").reset_index(drop=True)
SENSOR_COLS = [c for c in df.columns if c.startswith("sensor_")]
for c in SENSOR_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

## 2 · Preprocessing

In [ ]:
y_all = (df["label"].to_numpy() == 1).astype(np.int64)
X_all = df[SENSOR_COLS].copy()

n = len(df)
test_start = int(n * 0.80)
train_end = int(test_start * 0.80)  # 64% train, 16% validation, 20% test

if train_end <= 0 or test_start <= train_end or test_start >= n:
    raise ValueError(
        "Dataset too small for chronological train/validation/test split."
    )

X_train_df = X_all.iloc[:train_end].copy()
y_train = y_all[:train_end]

X_val_df = X_all.iloc[train_end:test_start].copy()
y_val = y_all[train_end:test_start]

X_test_df = X_all.iloc[test_start:].copy()
y_test = y_all[test_start:]

print("Rows:", n, "| Sensors:", len(SENSOR_COLS))

for name, yy in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test),
]:
    print(
        f"{name}: {len(yy)} rows, "
        f"defects={int(yy.sum())}, "
        f"prevalence={yy.mean():.2%}"
    )

print(
    "Time boundaries:",
    df.timestamp.iloc[0],
    "->",
    df.timestamp.iloc[train_end - 1],
    "| validation end:",
    df.timestamp.iloc[test_start - 1],
    "| test start:",
    df.timestamp.iloc[test_start],
)

assert df.timestamp.iloc[train_end - 1] <= df.timestamp.iloc[train_end]
assert df.timestamp.iloc[test_start - 1] <= df.timestamp.iloc[test_start]

BATCH_SIZE = 64

Rows: 1567 | Sensors: 590
Train: 1002 rows, defects=76, prevalence=7.58%
Validation: 251 rows, defects=11, prevalence=4.38%
Test: 314 rows, defects=17, prevalence=5.41%
Time boundaries: 2008-07-19 11:55:00 -> 2008-09-22 00:28:00 | validation end: 2008-10-02 19:25:00 | test start: 2008-10-02 20:54:00


In [ ]:
# 3. Causality-safe PS-L preprocessing

LEGACY_CODES = (100.0, 200.0, 300.0)
MISSING_THRESHOLD = 0.40
N_FIRMWARE_CATEGORIES = 5


def prepare_raw_features(frame):
    out = frame.copy()

    code = out["sensor_1"]
    legacy = code.isin(LEGACY_CODES)

    firmware = (
        code.map({100.0: 1, 200.0: 2, 300.0: 3})
        .fillna(0)
        .astype(np.int64)
    )
    firmware.loc[code.isna()] = 4

    sentinel = legacy & out["sensor_0"].eq(0.0)

    out.loc[sentinel, "sensor_0"] = np.nan
    out.loc[legacy, "sensor_1"] = np.nan

    return (
        out,
        firmware.to_numpy(dtype=np.int64),
        int(legacy.sum()),
        int(sentinel.sum()),
    )


def fit_preprocessor(X_fit_df):
    Xp, fw, legacy_n, sentinel_n = prepare_raw_features(X_fit_df)

    missing_rate = Xp.isna().mean()
    keep_cols = missing_rate[
        missing_rate <= MISSING_THRESHOLD
    ].index.tolist()

    if not keep_cols:
        raise ValueError(
            "No continuous features remain after missingness filtering."
        )

    imputer = SimpleImputer(
        strategy="median",
        keep_empty_features=True,
    )

    A_imp = imputer.fit_transform(
        Xp[keep_cols].to_numpy(dtype=np.float64)
    )

    scaler = StandardScaler()
    A_scaled = scaler.fit_transform(A_imp).astype(np.float32)

    state = {
        "keep_cols": keep_cols,
        "imputer": imputer,
        "scaler": scaler,
    }

    fw_onehot = np.eye(
        N_FIRMWARE_CATEGORIES,
        dtype=np.float32,
    )[fw]

    X_model = np.concatenate(
        [A_scaled, fw_onehot],
        axis=1,
    ).astype(np.float32)

    return (
        X_model,
        A_scaled,
        fw,
        state,
        {
            "legacy_rows": legacy_n,
            "sensor0_sentinels": sentinel_n,
        },
    )


def transform_preprocessor(X_df, state):
    Xp, fw, _, _ = prepare_raw_features(X_df)

    A = Xp[state["keep_cols"]].to_numpy(dtype=np.float64)
    A_imp = state["imputer"].transform(A)
    A_scaled = state["scaler"].transform(A_imp).astype(np.float32)

    fw_onehot = np.eye(
        N_FIRMWARE_CATEGORIES,
        dtype=np.float32,
    )[fw]

    X_model = np.concatenate(
        [A_scaled, fw_onehot],
        axis=1,
    ).astype(np.float32)

    return X_model, A_scaled, fw


# Fit preprocessing on training data only

X_train, X_train_cont, fw_train, prep_state, prep_diag = (
    fit_preprocessor(X_train_df)
)

X_val, X_val_cont, fw_val = transform_preprocessor(
    X_val_df, prep_state
)

X_test, X_test_cont, fw_test = transform_preprocessor(
    X_test_df, prep_state
)


# Display preprocessing diagnostics

print("Training-only preprocessing diagnostics:", prep_diag)
print("Continuous features retained:", len(prep_state["keep_cols"]))

print(
    "Input dimension (continuous + 5 one-hot firmware columns):",
    X_train.shape[1],
)

print(
    "Feature shapes:",
    X_train.shape,
    X_val.shape,
    X_test.shape,
)


# Validate dimensions and finite values

assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1]

assert (
    X_train_cont.shape[1]
    == X_val_cont.shape[1]
    == X_test_cont.shape[1]
)

assert (
    np.isfinite(X_train).all()
    and np.isfinite(X_val).all()
    and np.isfinite(X_test).all()
)

Training-only preprocessing diagnostics: {'legacy_rows': 85, 'sensor0_sentinels': 85}
Continuous features retained: 562
Input dimension (continuous + 5 one-hot firmware columns): 567
Feature shapes: (1002, 567) (251, 567) (314, 567)


## 3 · Tensors & loaders

In [ ]:
# 5. Training configuration

BATCH_SIZE = 64
EPOCHS = 100
LR = 5e-4
WEIGHT_DECAY = 1e-4

HIDDEN = 256
LATENT = 32

LAMBDA_CLF = 2.0  # tune only using validation data
LAMBDA_FW = 0.2   # auxiliary categorical reconstruction weight

PATIENCE = 15


# Create a DataLoader for model inputs, reconstruction targets,
# firmware categories, and defect labels.

def make_loader(X, X_cont, fw, y, shuffle=False):
    xt = torch.as_tensor(X, dtype=torch.float32)
    xct = torch.as_tensor(X_cont, dtype=torch.float32)
    fwt = torch.as_tensor(fw, dtype=torch.long).reshape(-1)
    yt = torch.as_tensor(y, dtype=torch.float32).reshape(-1)

    assert len(xt) == len(xct) == len(fwt) == len(yt)

    generator = (
        torch.Generator().manual_seed(SEED)
        if shuffle else None
    )

    ds = TensorDataset(xt, xct, fwt, yt)

    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        drop_last=False,
        generator=generator,
    )


# Initialize data loaders

train_loader = make_loader(
    X_train, X_train_cont, fw_train, y_train, shuffle=True
)

train_eval_loader = make_loader(
    X_train, X_train_cont, fw_train, y_train
)

val_loader = make_loader(
    X_val, X_val_cont, fw_val, y_val
)

test_loader = make_loader(
    X_test, X_test_cont, fw_test, y_test
)

## 4 · Model — supervised autoencoder

In [ ]:
class WaferSAE(nn.Module):
    def __init__(self, d_in, d_cont, hidden=256, latent=32, n_fw=5):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(0.10),

            nn.Linear(hidden, hidden // 2),
            nn.LayerNorm(hidden // 2),
            nn.GELU(),

            nn.Linear(hidden // 2, latent)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent, hidden // 2),
            nn.GELU(),

            nn.Linear(hidden // 2, hidden),
            nn.GELU(),

            nn.Linear(hidden, d_cont)
        )

        self.firmware_head = nn.Linear(latent, n_fw)
        self.defect_head = nn.Linear(latent, 1)

    def forward(self, x):
        if x.ndim != 2:
            raise ValueError(
                f"Expected input [B,F], got {tuple(x.shape)}"
            )

        z = self.encoder(x)

        reconstruction = self.decoder(z)               # [B, continuous_features]
        firmware_logits = self.firmware_head(z)        # [B, 5]
        defect_logits = self.defect_head(z).squeeze(-1) # [B]

        return reconstruction, firmware_logits, defect_logits, z

In [ ]:
model = WaferSAE(
    X_train.shape[1],
    X_train_cont.shape[1],
    HIDDEN,
    LATENT,
    N_FIRMWARE_CATEGORIES
).to(DEVICE)

In [ ]:

positive = max(int(y_train.sum()), 1)
negative = max(int((y_train == 0).sum()), 1)

pos_weight = torch.tensor(
    [negative / positive],
    dtype=torch.float32,
    device=DEVICE
)

bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
ce_firmware = nn.CrossEntropyLoss()


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

In [ ]:
model.train()

xb, xcb, fwb, yb = next(iter(train_loader))
xb, xcb, fwb, yb = (
    xb.to(DEVICE),
    xcb.to(DEVICE),
    fwb.to(DEVICE),
    yb.to(DEVICE).reshape(-1),
)

optimizer.zero_grad(set_to_none=True)

rec, fw_logits, defect_logits, _ = model(xb)

assert rec.shape == xcb.shape, (rec.shape, xcb.shape)
assert fw_logits.shape == (len(yb), N_FIRMWARE_CATEGORIES)
assert defect_logits.shape == yb.shape, (defect_logits.shape, yb.shape)

loss_check = (
    ((rec - xcb) ** 2).mean()
    + LAMBDA_FW * ce_firmware(fw_logits, fwb)
    + LAMBDA_CLF * bce(defect_logits, yb)
)

loss_check.backward()

for name, module in [
    ("encoder", model.encoder),
    ("continuous_decoder", model.decoder),
    ("firmware_head", model.firmware_head),
    ("defect_head", model.defect_head),
]:
    grad_norm = sum(
        float(p.grad.detach().norm())
        for p in module.parameters()
        if p.grad is not None
    )
    print(f"{name} gradient norm: {grad_norm:.6g}")
    assert grad_norm > 0, f"No gradients reached {name}"

optimizer.zero_grad(set_to_none=True)

print(
    "Shape/gradient checks passed.",
    "input", tuple(xb.shape),
    "continuous reconstruction", tuple(rec.shape),
    "firmware logits", tuple(fw_logits.shape),
    "defect logits", tuple(defect_logits.shape),
    "labels", tuple(yb.shape),
)

encoder gradient norm: 15.2752
continuous_decoder gradient norm: 0.0267968
firmware_head gradient norm: 0.392992
defect_head gradient norm: 3.15879
Shape/gradient checks passed. input (64, 567) continuous reconstruction (64, 562) firmware logits (64, 5) defect logits (64,) labels (64,)


## 5 · Training

In [ ]:
def collect_predictions(loader):
    model.eval()
    prob_chunks, label_chunks = [], []

    with torch.no_grad():
        for xb, xcb, fwb, yb in loader:
            xb = xb.to(DEVICE)

            _, _, logits, _ = model(xb)

            if logits.ndim != 1 or logits.shape[0] != yb.reshape(-1).shape[0]:
                raise ValueError(
                    f"Bad logits/label shapes: "
                    f"{tuple(logits.shape)} / {tuple(yb.shape)}"
                )

            prob_chunks.extend(
                torch.sigmoid(logits).cpu().numpy().tolist()
            )
            label_chunks.extend(yb.numpy().astype(int).tolist())

    return np.asarray(prob_chunks), np.asarray(label_chunks)


def run_validation_losses(loader):
    model.eval()
    recon_total = fw_total = clf_total = 0.0
    count = 0

    with torch.no_grad():
        for xb, xcb, fwb, yb in loader:
            xb, xcb = xb.to(DEVICE), xcb.to(DEVICE)
            fwb, yb = fwb.to(DEVICE), yb.to(DEVICE).reshape(-1)

            rec, fw_logits, logits, _ = model(xb)

            assert rec.shape == xcb.shape and logits.shape == yb.shape

            b = len(yb)

            recon_total += float(((rec - xcb) ** 2).mean()) * b
            fw_total += float(ce_firmware(fw_logits, fwb)) * b
            clf_total += float(bce(logits, yb)) * b

            count += b

    return recon_total / count, fw_total / count, clf_total / count

In [ ]:
history = []

best_val_ap = -np.inf
best_state = None
best_epoch = None
bad_epochs = 0

In [ ]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_sum = recon_sum = fw_sum = clf_sum = 0.0
    n_seen = 0

    for xb, xcb, fwb, yb in train_loader:
        xb, xcb = xb.to(DEVICE), xcb.to(DEVICE)
        fwb, yb = fwb.to(DEVICE), yb.to(DEVICE).reshape(-1)

        optimizer.zero_grad(set_to_none=True)

        rec, fw_logits, logits, _ = model(xb)

        assert rec.shape == xcb.shape
        assert fw_logits.shape == (len(yb), N_FIRMWARE_CATEGORIES)
        assert logits.shape == yb.shape

        recon_loss = ((rec - xcb) ** 2).mean()
        fw_loss = ce_firmware(fw_logits, fwb)
        clf_loss = bce(logits, yb)

        loss = (
            recon_loss
            + LAMBDA_FW * fw_loss
            + LAMBDA_CLF * clf_loss
        )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite loss at epoch {epoch}"
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )
        optimizer.step()

        b = len(yb)
        total_sum += float(loss.detach()) * b
        recon_sum += float(recon_loss.detach()) * b
        fw_sum += float(fw_loss.detach()) * b
        clf_sum += float(clf_loss.detach()) * b
        n_seen += b

    # Validation happens after all training batches for this epoch.
    val_recon, val_fw, val_clf = run_validation_losses(val_loader)
    val_probs_epoch, val_labels_epoch = collect_predictions(val_loader)

    if len(np.unique(val_labels_epoch)) < 2:
        raise ValueError(
            "Validation period must contain both classes "
            "to select a checkpoint by AP."
        )

    val_ap = average_precision_score(
        val_labels_epoch, val_probs_epoch
    )

    row = {
        "epoch": epoch,
        "train_total": total_sum / n_seen,
        "train_reconstruction": recon_sum / n_seen,
        "train_firmware_ce": fw_sum / n_seen,
        "train_classification": clf_sum / n_seen,
        "val_reconstruction": val_recon,
        "val_firmware_ce": val_fw,
        "val_classification": val_clf,
        "val_average_precision": float(val_ap),
    }
    history.append(row)

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:03d}: "
            f"train={row['train_total']:.4f}, "
            f"val_cls={val_clf:.4f}, "
            f"val_AP={val_ap:.4f}"
        )

    if val_ap > best_val_ap + 1e-5:
        best_val_ap = float(val_ap)
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        bad_epochs = 0
    else:
        bad_epochs += 1

        if bad_epochs >= PATIENCE:
            print("Early stopping at epoch", epoch)
            break


Epoch 001: train=3.5542, val_cls=1.0079, val_AP=0.0441
Epoch 005: train=1.0432, val_cls=2.7250, val_AP=0.0429
Epoch 010: train=0.7454, val_cls=4.0690, val_AP=0.0401
Epoch 015: train=0.6420, val_cls=4.3846, val_AP=0.0381
Early stopping at epoch 16


In [ ]:

# 5E. Restore best checkpoint and display training history

if best_state is None:
    raise RuntimeError("Training did not produce a valid checkpoint.")

model.load_state_dict(best_state)

print(f"Selected epoch {best_epoch} by validation AP={best_val_ap:.4f}")

history_df = pd.DataFrame(history)
display(history_df.tail())


Selected epoch 1 by validation AP=0.0441


,epoch,train_total,train_reconstruction,train_firmware_ce,train_classification,val_reconstruction,val_firmware_ce,val_classification,val_average_precision
11,12,0.703712,0.651498,0.236243,0.002482,49.336959,0.604048,4.153948,0.038714
12,13,0.682902,0.640278,0.193220,0.001990,49.323399,0.615488,4.202981,0.039276
13,14,0.661132,0.627279,0.152298,0.001697,49.311953,0.654171,4.284972,0.039424
14,15,0.641990,0.612122,0.126698,0.002264,49.301438,0.631392,4.384573,0.038096
15,16,0.628592,0.602173,0.105920,0.002618,49.294037,0.662772,4.434458,0.038823


## 6 · Evaluation

In [ ]:
val_probs, val_labels = collect_predictions(val_loader)

if len(np.unique(val_labels)) < 2:
    raise ValueError(
        "Validation period has only one class; cannot select a threshold."
    )

candidates = np.unique(np.concatenate(([0.0, 1.0], val_probs)))

threshold_rows = []

for threshold in candidates:
    pred = (val_probs >= threshold).astype(int)

    threshold_rows.append(
        (
            float(threshold),
            f1_score(val_labels, pred, zero_division=0),
            fbeta_score(val_labels, pred, beta=2, zero_division=0),
            precision_score(val_labels, pred, zero_division=0),
            recall_score(val_labels, pred, zero_division=0),
        )
    )

(
    best_threshold,
    best_val_f1,
    best_val_f2,
    best_val_precision,
    best_val_recall,
) = max(threshold_rows, key=lambda x: (x[1], x[2], x[0]))

print(f"Validation AP: {average_precision_score(val_labels, val_probs):.4f}")
print(f"Validation-selected threshold: {best_threshold:.6f}")

print(
    f"Validation F1={best_val_f1:.4f}, "
    f"F2={best_val_f2:.4f}, "
    f"precision={best_val_precision:.4f}, "
    f"recall={best_val_recall:.4f}"
)

print("Threshold and checkpoint selection used validation labels only.")

Validation AP: 0.0441
Validation-selected threshold: 0.417607
Validation F1=0.1119, F2=0.2273, precision=0.0606, recall=0.7273
Threshold and checkpoint selection used validation labels only.


In [ ]:
test_probs, test_labels = collect_predictions(test_loader)
test_preds = (test_probs >= best_threshold).astype(int)

precision = precision_score(test_labels, test_preds, zero_division=0)
recall = recall_score(test_labels, test_preds, zero_division=0)
f1 = f1_score(test_labels, test_preds, zero_division=0)
f2 = fbeta_score(test_labels, test_preds, beta=2, zero_division=0)

ap = average_precision_score(test_labels, test_probs)
cm = confusion_matrix(test_labels, test_preds, labels=[0, 1])
prevalence = float(test_labels.mean())

print(f"Selected checkpoint epoch: {best_epoch}")

print(f"Test samples: {len(test_labels)}")
print(f"Defective wafers: {int(test_labels.sum())}")
print(f"Natural test defect prevalence: {prevalence:.4%}")

print(f"Validation-selected threshold: {best_threshold:.6f}")

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")
print(f"F2: {f2:.4f}")
print(f"Average precision (AP): {ap:.4f}")

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(cm)

print(
    classification_report(
        test_labels,
        test_preds,
        labels=[0, 1],
        target_names=["Passing", "Defective"],
        zero_division=0,
    )
)

Selected checkpoint epoch: 1
Test samples: 314
Defective wafers: 17
Natural test defect prevalence: 5.4140%
Validation-selected threshold: 0.417607
Precision: 0.1222
Recall: 0.6471
F1: 0.2056
F2: 0.3481
Average precision (AP): 0.1258
Confusion matrix [[TN, FP], [FN, TP]]:
[[218  79]
 [  6  11]]
              precision    recall  f1-score   support

     Passing       0.97      0.73      0.84       297
   Defective       0.12      0.65      0.21        17

    accuracy                           0.73       314
   macro avg       0.55      0.69      0.52       314
weighted avg       0.93      0.73      0.80       314



## 7 · Release gate

In [ ]:
metrics = {
    "test_samples": int(len(test_labels)),
    "test_defects": int(test_labels.sum()),
    "test_prevalence": prevalence,

    "selected_epoch_by_validation_ap": int(best_epoch),
    "validation_average_precision_for_selected_epoch": float(best_val_ap),
    "threshold_selected_on_validation": float(best_threshold),

    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "f2": float(f2),
    "average_precision": float(ap),
    "confusion_matrix_labels_0_1": cm.tolist(),

    "retained_continuous_features": int(len(prep_state["keep_cols"])),
    "training_rows": int(len(y_train)),
    "validation_rows": int(len(y_val)),
    "test_rows": int(len(y_test)),

    "lambda_classification": float(LAMBDA_CLF),
    "lambda_firmware": float(LAMBDA_FW),
}

metrics_path = "/content/wafer_guard_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("\nSaved measured metrics to:", metrics_path)


Saved measured metrics to: /content/wafer_guard_metrics.json
